<a href="https://colab.research.google.com/github/motasimfadul/cosc726-motasim-fadul/blob/main/week01/lab0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COSC726 · Lab 0 — Engineering Baseline & Trace Literacy

**Agentic Artificial Intelligence · Week 1 · 2-hour supervised lab**

Two goals, and the second is the one that lasts:

1. **A working, reproducible setup** — environment verified, starter repo cloned, CI green on your first push
2. **Trace literacy** — read a real agent trace closely enough to audit its evidence, its permissions, and its stopping decision

The trace is **pre-recorded and deterministic**. No model, API key, or paid account is required. The `DECIDE`
entries are *authored rationale summaries* for teaching and audit — they are **not** private model reasoning.

> **Running example:** the customer-support agent helping **Layla** with order **#A1032**. It follows us all
> term, gaining one capability each week.

**Where you work:** this notebook runs in **Google Colab** (or locally, if you prefer).
**What you submit:** push this executed notebook to `week01/` in your repo, then tag `week-01-complete`.

## Part A · Environment check (~15 min)

A `WARN` is not a failure — it tells you what to repair or report **before you leave today**. Setup problems
compound silently, so today is the cheapest possible day to fix them.

In [10]:
from __future__ import annotations

import importlib.util
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

print(f"Python {platform.python_version()} on {platform.system()} ({platform.machine()})")
print("PASS · Python 3.11+" if sys.version_info >= (3, 11)
      else "WARN · install Python 3.11 or newer before Lab 1")
print(f"Environment: {'Google Colab (managed)' if IN_COLAB else 'local machine'}")
print(f"Working directory: {Path.cwd()}")

Python 3.12.13 on Linux (x86_64)
PASS · Python 3.11+
Environment: Google Colab (managed)
Working directory: /content


In [11]:
# Isolation check. Colab already gives you a clean managed runtime, so a venv is
# only expected when you are running locally.
if IN_COLAB:
    print("PASS · Colab runtime is isolated and disposable — no virtualenv needed here.")
    print("       Note: Colab RESETS between sessions. GitHub is your permanent record.")
    in_venv = True          # not applicable in Colab; treated as satisfied
else:
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    if in_venv:
        print(f"PASS · virtual environment active: {sys.prefix}")
    else:
        print("WARN · no virtual environment detected")
        if platform.system() == "Windows":
            print(r"      create: python -m venv .venv    activate: .venv\Scripts\activate")
        else:
            print("      create: python3 -m venv .venv    activate: source .venv/bin/activate")

PASS · Colab runtime is isolated and disposable — no virtualenv needed here.
       Note: Colab RESETS between sessions. GitHub is your permanent record.


In [12]:
def run_text(command: list[str]) -> str:
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return (result.stdout or result.stderr).strip()
    except OSError as exc:
        return f"ERROR: {exc}"

git_ok = shutil.which("git") is not None
if git_ok:
    print("PASS ·", run_text(["git", "--version"]))
    name = run_text(["git", "config", "user.name"])
    email = run_text(["git", "config", "user.email"])
    if name and email and not name.startswith("ERROR"):
        print(f"PASS · Git identity: {name} <{email}>")
    else:
        print("WARN · set your identity:")
        print('      git config --global user.name "Your Name"')
        print('      git config --global user.email "you@university.ac.uk"')
else:
    print("WARN · Git is not on PATH")

print("PASS · kernel executable:", sys.executable)

PASS · git version 2.34.1
WARN · set your identity:
      git config --global user.name "Your Name"
      git config --global user.email "you@university.ac.uk"
PASS · kernel executable: /usr/bin/python3


### The starter repo — you *inherit* the practices, you don't build them

Clone the classroom starter in a terminal (or a Colab cell prefixed with `!`). It already ships
`pytest`, linting, and a CI workflow **preconfigured** — your job this week is to get a green CI run, not
to assemble a toolchain from scratch.

```bash
git clone <classroom-starter-url> cosc726-<surname>
cd cosc726-<surname>
python -m pip install -r requirements.txt      # version-pinned — do NOT upgrade mid-term
python COSC726_W01_hello_agent_mock.py --self-test
```

Expected: `TRACE SELF-TEST PASSED`. That command is also what CI runs on your first push — so if it passes
locally and CI is red, the difference is your environment, and that is exactly what this week is about.

## Part B · Run the deterministic trace (~15 min)

Read it **twice**: first as a support story, then as a sequence of *observations, decisions, actions, and
state changes*. Watch especially for what the agent does **not** do.

In [13]:
MOCK_TRACE = [
    (1, "USER", "Hi, I'm Layla. Where is my order #A1032? It was meant to arrive Tuesday."),
    (2, "DECIDE", "I need the current order status. Use the read-only lookup_order tool."),
    (3, "ACT", 'lookup_order(order_id="A1032")'),
    (4, "OBSERVE", '{"order_id":"A1032","item":"desk lamp","status":"delayed at depot",'
                   '"promised_day":"Tuesday","new_eta":"Friday"}'),
    (5, "DECIDE", "The delay is three calendar days. Check the policy before mentioning compensation."),
    (6, "ACT", 'get_late_delivery_policy(topic="goodwill credit")'),
    (7, "OBSERVE", '{"eligibility":"delay of 3 or more calendar days",'
                   '"remedy":"10% goodwill credit","approval_required":true}'),
    (8, "DECIDE", "The order appears eligible. Applying a credit changes the account, so request "
                  "human approval rather than claim it happened."),
    (9, "ACT", 'request_approval(action="apply_goodwill_credit", order_id="A1032", amount_percent=10)'),
    (10, "OBSERVE", '{"approval_id":"APR-2048","status":"pending","account_changed":false}'),
    (11, "DECIDE", "I have verified the facts and opened the required approval request. Stop and "
                   "report the status, evidence, and next step."),
    (12, "RESPOND", "Hi Layla - order #A1032 is delayed at the depot and is now due Friday. Because "
                    "that is three calendar days late, it appears eligible for a 10% goodwill credit. "
                    "I submitted approval request APR-2048; no account change has been made yet."),
]

def play_trace(upto: int | None = None) -> None:
    events = MOCK_TRACE if upto is None else MOCK_TRACE[:upto]
    width = max(len(kind) for _, kind, _ in MOCK_TRACE)
    for step, kind, text in events:
        print(f"[{kind:<{width}}] step {step:>2}: {text}")
        print("-" * 108)

play_trace()

[USER   ] step  1: Hi, I'm Layla. Where is my order #A1032? It was meant to arrive Tuesday.
------------------------------------------------------------------------------------------------------------
[DECIDE ] step  2: I need the current order status. Use the read-only lookup_order tool.
------------------------------------------------------------------------------------------------------------
[ACT    ] step  3: lookup_order(order_id="A1032")
------------------------------------------------------------------------------------------------------------
[OBSERVE] step  4: {"order_id":"A1032","item":"desk lamp","status":"delayed at depot","promised_day":"Tuesday","new_eta":"Friday"}
------------------------------------------------------------------------------------------------------------
[DECIDE ] step  5: The delay is three calendar days. Check the policy before mentioning compensation.
----------------------------------------------------------------------------------------------------

**The move to notice:** at step 8 the agent decides it is *eligible* for a credit — and then, at step 9,
**does not apply it**. It opens an approval request instead, and step 10 records `"account_changed": false`.
The final message says the credit "appears eligible" and that no change has been made. That restraint is the
difference between Level 3 and Level 4 on the lecture's autonomy spectrum, and it is designed, not accidental.

## Part C · Classify the loop (~25 min)

Label **all 12 events**:

| Phase | Meaning |
|---|---|
| `sense` | receive information from the environment |
| `reason` | decide or plan the next step |
| `act` | call a tool, request approval, change state, or respond |
| `observe` | register the result of an action and update task state |

Steps **4, 7 and 10** accept either `sense` or `observe` — in software agents a tool result is often both the
outcome of the last action and the next perception. Choose one and be ready to defend it.

In [14]:
my_annotations = {
    1: "sense", 2: "reason", 3: "act", 4: "observe",
    5: "reason", 6: "act", 7: "observe", 8: "reason",
    9: "act", 10: "observe", 11: "reason", 12: "act",
}
# Steps 4, 7 and 10 are tool RESULTS being registered into task state, so they
# are classified here as "observe" rather than "sense" — "sense" is reserved
# for the step-1 USER message, the system's only perception that originates
# outside the agent's own action loop. Either answer is defensible; the
# argument, not the label, is the point.

ANSWER_KEY = {1: ["sense"], 2: ["reason"], 3: ["act"], 4: ["observe", "sense"],
              5: ["reason"], 6: ["act"], 7: ["observe", "sense"], 8: ["reason"],
              9: ["act"], 10: ["observe", "sense"], 11: ["reason"], 12: ["act"]}

def check_annotations(answers: dict[int, str]) -> int:
    correct = 0
    for step, expected_list in ANSWER_KEY.items():
        expected = set(expected_list)
        got = str(answers.get(step, "")).strip().lower()
        ok = got in expected
        correct += int(ok)
        verdict = "OK" if ok else ("UNANSWERED" if got in {"", "..."}
                                   else "EXPECTED " + " or ".join(sorted(expected)))
        print(f"step {step:>2}: {got or '(blank)':<10} {verdict}")
    print(f"\nScore: {correct}/{len(ANSWER_KEY)}")
    return correct

check_annotations(my_annotations)


step  1: sense      OK
step  2: reason     OK
step  3: act        OK
step  4: observe    OK
step  5: reason     OK
step  6: act        OK
step  7: observe    OK
step  8: reason     OK
step  9: act        OK
step 10: observe    OK
step 11: reason     OK
step 12: act        OK

Score: 12/12


12

Do the same in `COSC726_W01_hello_agent_mock.py` (edit its `ANNOTATIONS` dict), then verify:

```bash
python COSC726_W01_hello_agent_mock.py --check      # target: 12/12
```
In Colab, prefix with `!`. Both the notebook and the script must reach 12/12.

## Part D · Evidence audit (~20 min)

Replace every `TODO`. Two or three precise sentences each; **cite step numbers** as evidence.

In [15]:
TRACE_AUDIT = """
Name: Motasim Fadul
Student ID: 12345678
--------------------------------------------------------------------------
1. EVIDENCE. Which observations support the final response? Cite step numbers.
The final response (step 12) rests on three observed facts. Step 4 supplies
the order's current status and revised ETA (delayed at depot, new ETA
Friday). Step 7 supplies the policy threshold and remedy (3+ calendar days
late qualifies for a 10% goodwill credit, approval required). Step 10
supplies the outcome of the approval request itself (APR-2048, pending,
account_changed=false). Every claim in step 12 traces back to one of these
three OBSERVE events; nothing in the reply is invented.

2. PERMISSIONS. Which tools are read-only, and which action could lead to an
   account change? Why is the approval gate appropriate?
lookup_order (step 3) and get_late_delivery_policy (step 6) are read-only
queries: they retrieve information and change nothing. request_approval
(step 9) is the tool that touches the account-changing path, since its
purpose is to initiate a credit — but it only *opens a request*, it does not
apply the credit itself. The approval gate is appropriate because a credit
is a consequential, account-changing action; per the "tool schema is not
authority to use the tool" principle, a read-only lookup and a financial
remedy must not carry the same authority, so the agent is only permitted to
propose the change and must wait for a human or policy check to execute it.

3. STATE CHANGE. What evidence shows the credit has NOT yet been applied?
Step 10's observation explicitly records "account_changed": false alongside
a "pending" status for approval APR-2048. Step 12 restates this in plain
language ("no account change has been made yet"). There is no ACT event
anywhere in the trace that calls an apply_credit or execute-type tool, so no
successful write to the account ever occurs.

4. TERMINATION. Why is step 11 a valid stop decision? Name one condition that
   should instead cause escalation or another loop iteration.
Step 11 is a valid stop because the agent has completed everything within
its delegated authority: it has verified the order status, checked policy
eligibility, and opened the one action it is permitted to take (an approval
request) — there is no further step it could take without exceeding its
authority or fabricating an outcome. It is correct to stop and report,
rather than loop, wait synchronously, or guess at the approval's outcome.
By contrast, if request_approval (step 9) had returned an error or a
timeout instead of a valid approval ID, the agent should escalate to a
human agent rather than silently continuing or fabricating a pending state.

5. TRACE LIMITS. Why should DECIDE be treated as a designed rationale summary,
   not as guaranteed access to private model reasoning?
The lecture notes and lab brief are explicit that DECIDE lines are authored,
concise rationale summaries written for teaching and audit, produced by
whoever designed the trace (or, in a live system, by prompting the model to
narrate a summary) — they are not a faithful readout of the model's
internal computation. Treating them as guaranteed access to "true" reasoning
would be an unsupported inference: the trace records what was written down,
not what actually happened inside the model, and a concise justification is
compatible with many different underlying (or absent) reasoning processes.
"""

remaining_audit = TRACE_AUDIT.count("TODO")
print(TRACE_AUDIT)
print("PASS · trace audit complete" if remaining_audit == 0
      else f"WARN · {remaining_audit} TODO item(s) remain")



Name: Motasim Fadul
Student ID: 12345678
--------------------------------------------------------------------------
1. EVIDENCE. Which observations support the final response? Cite step numbers.
The final response (step 12) rests on three observed facts. Step 4 supplies
the order's current status and revised ETA (delayed at depot, new ETA
Friday). Step 7 supplies the policy threshold and remedy (3+ calendar days
late qualifies for a 10% goodwill credit, approval required). Step 10
supplies the outcome of the approval request itself (APR-2048, pending,
account_changed=false). Every claim in step 12 traces back to one of these
three OBSERVE events; nothing in the reply is invented.

2. PERMISSIONS. Which tools are read-only, and which action could lead to an
   account change? Why is the approval gate appropriate?
lookup_order (step 3) and get_late_delivery_policy (step 6) are read-only
queries: they retrieve information and change nothing. request_approval
(step 9) is the tool that tou

## Part E · PEAS and autonomy (~15 min)

Specify the system. Use the trace as evidence — do not answer from intuition alone. (This feeds directly
into the seminar's PEAS exercise.)

In [16]:
DESIGN_WORKSHEET = """
PEAS
----
Performance measure: Correct, evidence-grounded resolution of the customer's
    query; compliance with the late-delivery policy; no unauthorised
    account change (a hard constraint, not just a preference); reasonable
    latency and tool-call cost. A measure like "tickets closed per hour"
    is explicitly rejected because it can be maximised by closing
    unresolved tickets.

Environment:         The customer (Layla), the order/delivery-tracking
    system reached via lookup_order, the policy store reached via
    get_late_delivery_policy, and the human support/approval staff who
    action request_approval outcomes.

Actuators:           query order data (lookup_order, step 3), query policy
    (get_late_delivery_policy, step 6), request approval for a
    consequential action (request_approval, step 9), and send a reply to
    the customer (RESPOND, step 12).

Sensors:              the initial user message (step 1), tool-returned
    results for order status and policy (steps 4, 7), the approval
    system's status update (step 10), and implicit task state carried
    between steps (e.g. that eligibility has already been checked before
    approval is requested).

AUTONOMY AND RISK
-----------------
Starting autonomy level (1-4): Level 3 — Supervised autonomy. The agent
    gathers evidence and takes low-risk read actions on its own initiative
    (steps 3, 6), but the one action with financial/account consequence is
    not executed directly — it is only proposed, pending human sign-off.

Evidence from the trace (cite step numbers): Step 9 calls request_approval
    rather than an "apply_credit" action; step 10 confirms
    "account_changed": false with a "pending" status; step 12 reports the
    pending state honestly rather than claiming the credit was applied.
    This is exactly the Level 3 pattern: the system completes low-risk
    steps itself but gates the consequential one behind approval.

Highest-risk action in the trace: request_approval for a 10% goodwill
    credit (step 9) — it is the only step that touches money and an
    account, even though execution itself still requires a separate,
    human-gated step beyond this trace.

What evidence would justify moving up one level? A demonstrated,
    audited track record showing the policy check is reliably correct
    (accurate threshold application over many real cases), a defined cap
    on credit size/frequency so financial exposure per case is small and
    bounded, and an organisational decision to delegate that specific,
    low-value, well-verified action class to Level 4 — while almost
    certainly keeping other, higher-value or less-reversible actions
    gated. Per the risk rule, the highest-consequence action still sets
    the ceiling, so moving up should be scoped to this narrow action, not
    granted globally.
"""

remaining_peas = DESIGN_WORKSHEET.count("TODO")
print(DESIGN_WORKSHEET)
print("PASS · design worksheet complete" if remaining_peas == 0
      else f"WARN · {remaining_peas} TODO item(s) remain")



PEAS
----
Performance measure: Correct, evidence-grounded resolution of the customer's
    query; compliance with the late-delivery policy; no unauthorised
    account change (a hard constraint, not just a preference); reasonable
    latency and tool-call cost. A measure like "tickets closed per hour"
    is explicitly rejected because it can be maximised by closing
    unresolved tickets.

Environment:         The customer (Layla), the order/delivery-tracking
    system reached via lookup_order, the policy store reached via
    get_late_delivery_policy, and the human support/approval staff who
    action request_approval outcomes.

Actuators:           query order data (lookup_order, step 3), query policy
    (get_late_delivery_policy, step 6), request approval for a
    consequential action (request_approval, step 9), and send a reply to
    the customer (RESPOND, step 12).

Sensors:              the initial user message (step 1), tool-returned
    results for order status and polic

## Part F · Decision memo (~20 min)

**Every lab in this module closes with the same half-page memo.** It is formative — it carries no direct
marks — but it is the postgraduate standard, and it feeds your project report and the exam's design question.

This week the sharpest question is **reproducibility**: *could a marker clone your repository and re-run your
work unchanged?*

In [17]:
DECISION_MEMO = """
COSC726 · Lab 0 decision memo
Name: Motasim Fadul
--------------------------------------------------------------------------
1. BETTER THAN WHAT?  What is the baseline this agent should be compared with
   (e.g. a fixed auto-reply template)? What does the trace do that the baseline
   cannot?
The natural baseline is a fixed auto-reply template ("Thanks for reaching
out, we're looking into your order and will respond within 24 hours") or a
simple keyword-routing rule that just forwards the message to a queue. That
baseline cannot check the actual order status, cannot check whether the
delay crosses the policy's eligibility threshold, and cannot initiate the
correct next step (an approval request) — it can only stall. This trace
grounds every claim in retrieved evidence (steps 4, 7) and takes a concrete,
correctly-gated action (step 9) instead of a generic holding message.

2. MEASURED BY WHAT?  Propose ONE measurable criterion for whether this agent
   handled Layla well. State how you would compute it.
Grounded-response rate: the proportion of claims in the final RESPOND event
that are directly supported by a successful, logged OBSERVE event earlier
in the same trace. Compute it by parsing each RESPOND message into its
factual claims (status, ETA, eligibility, approval state) and checking each
one against the trace's OBSERVE log; a claim with no matching evidence
counts as unsupported, and the rate is (supported claims) / (total claims)
averaged across a sample of runs.

3. AT WHAT COST?  Count the tool calls and events. If each step involved a model
   call over a growing transcript, where would the cost grow fastest?
The trace has 12 events total and 3 tool calls (steps 3, 6, 9). If each
DECIDE step required a fresh model call over the full transcript so far
(rather than a fixed-size summary of state), cost would grow fastest in the
later DECIDE steps (5, 8, 11), since each one re-sends a longer history —
the growing context window, not the tool calls themselves, is the part that
scales badly as the trace lengthens.

4. UNDER WHAT FAILURE CONDITIONS?  Pick one step and describe what the agent
   should do if that tool returned an error or a timeout instead.
If step 9's request_approval call timed out or returned an error instead of
an approval ID, the agent should not report "APR-2048 pending" (that would
be an unsupported claim). It should retry once with backoff, and if it
still fails, tell Layla honestly that the credit request could not be
submitted yet and escalate to a human agent — rather than silently retrying
forever or guessing that approval succeeded.

5. HOW REPRODUCIBLE?  Could a marker clone your repo and reproduce today's
   output exactly? Name the two things most likely to break that.
Yes — the trace is a fixed, deterministic literal (no randomness, no
network, no API key), so any Python 3.11+ environment running this script
should print an identical trace. The two things most likely to break that:
(1) an unpinned or mismatched dependency/Python version between my machine
and the marker's, and (2) forgetting to push the edited
COSC726_W01_hello_agent_mock.py alongside the notebook, so the script's
ANNOTATIONS dict and --check output no longer match what the notebook
claims.

6. WHAT ENGINEERING DECISION FOLLOWS?  State one concrete change you would make
   before this agent went anywhere near a real customer.
I would enforce the read/propose/approve/execute separation at the tool
layer itself, not just in the prompt: give request_approval its own scoped
credential that has no path to an "apply_credit" execute-tool at all, so
that even a compromised or misbehaving controller cannot escalate a
proposal into an actual account change without a genuinely separate,
human-owned execute step.
"""

remaining_memo = DECISION_MEMO.count("TODO")
print(DECISION_MEMO)
print("PASS · decision memo complete" if remaining_memo == 0
      else f"WARN · {remaining_memo} TODO item(s) remain")



COSC726 · Lab 0 decision memo
Name: Motasim Fadul
--------------------------------------------------------------------------
1. BETTER THAN WHAT?  What is the baseline this agent should be compared with
   (e.g. a fixed auto-reply template)? What does the trace do that the baseline
   cannot?
The natural baseline is a fixed auto-reply template ("Thanks for reaching
out, we're looking into your order and will respond within 24 hours") or a
simple keyword-routing rule that just forwards the message to a queue. That
baseline cannot check the actual order status, cannot check whether the
delay crosses the policy's eligibility threshold, and cannot initiate the
correct next step (an approval request) — it can only stall. This trace
grounds every claim in retrieved evidence (steps 4, 7) and takes a concrete,
correctly-gated action (step 9) instead of a generic holding message.

2. MEASURED BY WHAT?  Propose ONE measurable criterion for whether this agent
   handled Layla well. State how you

## Part G · Submission readiness

Run this after completing Parts C–F.

In [18]:
score = check_annotations(my_annotations)

checks = {
    "Python 3.11+":               sys.version_info >= (3, 11),
    "Isolated environment":       in_venv,
    "Git available":              git_ok,
    "Annotations 12/12":          score == 12,
    "Trace audit complete":       TRACE_AUDIT.count("TODO") == 0,
    "PEAS / autonomy complete":   DESIGN_WORKSHEET.count("TODO") == 0,
    "Decision memo complete":     DECISION_MEMO.count("TODO") == 0,
}

print()
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'WARN'} · {label}")

print()
if all(checks.values()):
    print("READY TO SUBMIT — now follow the push steps in the next cell.")
else:
    print("Resolve the warnings above, or document the setup issue with a demonstrator.")

step  1: sense      OK
step  2: reason     OK
step  3: act        OK
step  4: observe    OK
step  5: reason     OK
step  6: act        OK
step  7: observe    OK
step  8: reason     OK
step  9: act        OK
step 10: observe    OK
step 11: reason     OK
step 12: act        OK

Score: 12/12

PASS · Python 3.11+
PASS · Isolated environment
PASS · Git available
PASS · Annotations 12/12
PASS · Trace audit complete
PASS · PEAS / autonomy complete
PASS · Decision memo complete

READY TO SUBMIT — now follow the push steps in the next cell.


### Push it (this is the submission)

1. **Restart & Run All** so the notebook is clean top-to-bottom.
2. **From Colab:** `File → Save a copy in GitHub` → repo `cosc726-<surname>`, path **`week01/lab0.ipynb`**,
   and write a real commit message (*"Lab 0: trace annotated 12/12, memo complete"* — not *"update"*).
3. **Also push** your edited `COSC726_W01_hello_agent_mock.py` to `week01/`.
4. **Tag the checkpoint:** on GitHub, *Releases → new tag* **`week-01-complete`**. This is your safety net —
   if a later week breaks, you can branch cleanly from here.
5. **Check CI is green** on the push. A red CI is a finding, not a disaster — but report it today.

**Deadline:** pushed before the Week 2 lecture. The teaching team reads your repository directly; there is no
separate upload. Your repo link belongs in the Moodle *Repository link* box (once, in Week 1).

> **Never commit secrets.** Nothing this week needs an API key — but the habit starts now: real keys go in a
> git-ignored `.env`, and only `.env.example` is committed.

## Stretch · Predict, then structure

1. Run `play_trace(upto=4)` — stop right after the first tool result — and **write down your prediction** of
   steps 5–12 before revealing them. Did you predict the *approval gate*, or did you expect the agent to
   apply the credit itself?
2. Run `!python COSC726_W01_hello_agent_mock.py --json` and look at the trace as structured data.
3. Propose **four fields** an observability system should add — for example timestamp, run ID, actor, tool
   latency, permission class, state delta, or stop reason. You will meet the real version of this list in
   Week 2 (telemetry) and wire it into CI in Week 11.

---
**Next week:** LLM foundations from zero — tokens, context budgets, message roles, sampling, and the
difference between model output and system behaviour.